# **Komparasi Algoritma Apriori, ECLAT, dan FP-Growth dalam Menemukan Pola Pembelian pada Dataset Online Retail**

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')         
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time, tracemalloc, os, warnings
warnings.filterwarnings('ignore')

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder

In [2]:
# ── ECLAT manual 
def run_eclat(basket_bool, min_support):
    """
    Implementasi ECLAT dengan vertical TID-list intersection.
    Tidak membutuhkan pyECLAT — murni Python + pandas.
    Input : basket_bool — DataFrame boolean (baris=transaksi, kolom=item)
    Output: dict {frozenset(items): support_float}
    """
    n_trans = len(basket_bool)
    min_count = int(np.ceil(min_support * n_trans))

    # Buat TID-list tiap item (hanya item yang memenuhi min_support)
    tid_list = {}
    for col in basket_bool.columns:
        tids = frozenset(basket_bool.index[basket_bool[col]])
        if len(tids) >= min_count:
            tid_list[frozenset([col])] = tids

    frequent = {k: len(v)/n_trans for k, v in tid_list.items()}

    # Generate k-itemset via TID intersection
    current_level = dict(tid_list)
    while current_level:
        next_level = {}
        items_list = list(current_level.keys())
        for i in range(len(items_list)):
            for j in range(i+1, len(items_list)):
                a, b = items_list[i], items_list[j]
                union = a | b
                if len(union) != len(a) + 1:
                    continue
                inter = current_level[a] & current_level[b]
                if len(inter) >= min_count:
                    next_level[union] = inter
                    frequent[union] = len(inter) / n_trans
        current_level = next_level
    return frequent

def eclat_to_df(frequent_dict):

    rows = []

    for itemset, support in frequent_dict.items():
        rows.append({
            'support': support,
            'itemsets': frozenset(itemset)
        })

    return pd.DataFrame(rows)

#penyimpanan visualisasi 
os.makedirs('output_plots', exist_ok=True)
 
C = {
    'primary'  : '#1A3C5E',
    'secondary': '#2E86AB',
    'accent'   : '#F4A261',
    'highlight': '#E76F51',
    'green'    : '#2A9D8F',
    'grid'     : '#E0E0E0',
    'apriori'  : '#E76F51',
    'eclat'    : '#2E86AB',
    'fpgrowth' : '#2A9D8F',
}

plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.color'       : C['grid'],
    'grid.linewidth'   : 0.6,
    'figure.dpi'       : 150,
})

def savefig(name):
    plt.savefig(f'output_plots/{name}', bbox_inches='tight', facecolor='white')
    plt.close()
    print(f"  Tersimpan: output_plots/{name}")

## Load Data

Dataset yang digunakan dalam penelitian ini adalah Online Retail Dataset yang diperoleh dari Kaggle melalui tautan berikut: https://www.kaggle.com/datasets/tunguz/online-retail. 

Dataset ini berisi data transaksi penjualan sebuah perusahaan retail online yang menjual berbagai produk kebutuhan rumah tangga dan souvenir kepada pelanggan dari berbagai negara. 

Dataset terdiri dari sekitar 541.909 data transaksi dengan 8 atribut utama yaitu invoiceNo, StockCode, description, Quantity, InvoiceData, UnitPrice, CustomerID, dan Country. 

Data transaksi dataset ini dari periode 1 Desember 2010 sampai 9 desember 2011. 

In [3]:
print("1. LOAD DATA")

df_raw = pd.read_csv('Online_Retail.csv', encoding='latin-1')
df_raw['InvoiceDate'] = pd.to_datetime(df_raw['InvoiceDate'])

print(f"Shape raw          : {df_raw.shape}")
print(f"Transaksi unik     : {df_raw['InvoiceNo'].nunique():,}")
print(f"Negara unik        : {df_raw['Country'].nunique()}")
print(f"Periode            : {df_raw['InvoiceDate'].min().date()} → {df_raw['InvoiceDate'].max().date()}")
print("\nMissing Values:")
mv = df_raw.isnull().sum()
for col, n in mv[mv > 0].items():
    print(f"  {col:<15}: {n:>6,}  ({n/len(df_raw)*100:.2f}%)")

1. LOAD DATA
Shape raw          : (541909, 8)
Transaksi unik     : 25,900
Negara unik        : 38
Periode            : 2010-12-01 → 2011-12-09

Missing Values:
  Description    :  1,454  (0.27%)
  CustomerID     : 135,080  (24.93%)


## Preprocessing

Tahapan preprocessing:
1. Menghapus Data yang Memiliki Missing Value
2. Menghapus Transaksi Pembatalan 
3. Menghapus Nilai Quantity Bernilai Nol atau Negatif
4. Menghapus data duplikat
5. Menyaring data yang tidak relevan
6. Melakukan Normalisasi Nama Produk

In [4]:
print("2. PREPROCESSING")

steps = {'Raw': len(df_raw)}
df = df_raw.copy()

df = df[df['Country'] == 'United Kingdom']
steps['Filter UK'] = len(df)
print(f"[1] Filter UK only        : {len(df):>7,} baris")

df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
steps['Hapus Batal'] = len(df)
print(f"[2] Hapus transaksi batal : {len(df):>7,} baris")

df = df.dropna(subset=['Description'])
steps['Hapus NA'] = len(df)
print(f"[3] Hapus missing value   : {len(df):>7,} baris")

df = df[df['Quantity'] > 0]
steps['Qty > 0'] = len(df)
print(f"[4] Qty > 0               : {len(df):>7,} baris")

df = df[df['UnitPrice'] > 0]
steps['Price > 0'] = len(df)
print(f"[5] UnitPrice > 0         : {len(df):>7,} baris")

df['Description'] = df['Description'].str.strip().str.upper()
df = df.drop_duplicates()
steps['Hapus Duplikat'] = len(df)
print(f"[6] Hapus duplikat        : {len(df):>7,} baris")

df['Revenue']   = df['Quantity'] * df['UnitPrice']
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour']      = df['InvoiceDate'].dt.hour

df.to_csv('output_plots/cleaned_data.csv', index=False)
print(f"\n Data bersih: {len(df):,} baris | {df['InvoiceNo'].nunique():,} transaksi | {df['Description'].nunique():,} produk")

2. PREPROCESSING
[1] Filter UK only        : 495,478 baris
[2] Hapus transaksi batal : 487,622 baris
[3] Hapus missing value   : 486,168 baris
[4] Qty > 0               : 485,694 baris
[5] UnitPrice > 0         : 485,123 baris
[6] Hapus duplikat        : 479,985 baris

 Data bersih: 479,985 baris | 18,019 transaksi | 3,996 produk


## Analisis Data Eksploratif

In [5]:
print("3. EDA — VISUALISASI")

# ── Plot 1: Preprocessing Funnel 
fig, ax = plt.subplots(figsize=(10,6))
labels = list(steps.keys()); values = list(steps.values())
pal = [C['primary']] + [C['secondary']]*(len(labels)-2) + [C['green']]
bars = ax.barh(labels, values, color=pal, edgecolor='white', height=0.6)
for b, v in zip(bars, values):
    ax.text(b.get_width()+3000, b.get_y()+b.get_height()/2,
            f'{v:,}', va='center', fontsize=9, color=C['primary'], fontweight='bold')
ax.invert_yaxis()
ax.set_xlabel('Jumlah Baris', fontsize=11)
ax.set_title('Preprocessing Funnel — Online Retail UK', fontsize=14,
             fontweight='bold', pad=12, color=C['primary'])
ax.set_xlim(0, max(values)*1.14)
savefig('plot1_preprocessing_funnel.png')

# ── Plot 2: Top 15 Produk Terlaris (Qty) ─────────────────────
top15 = df.groupby('Description')['Quantity'].sum().sort_values(ascending=False).head(15)
short = [l[:38]+'…' if len(l)>38 else l for l in top15.index[::-1]]

fig, ax = plt.subplots(figsize=(12,7))
pal15 = [C['highlight'] if i >= 12 else C['secondary'] for i in range(15)]
ax.barh(short, top15.values[::-1], color=pal15, edgecolor='white', height=0.7)
for i, v in enumerate(top15.values[::-1]):
    ax.text(v+500, i, f'{v:,}', va='center', fontsize=8)
ax.set_xlabel('Total Quantity Terjual', fontsize=11)
ax.set_title('Top 15 Produk Terlaris berdasarkan Quantity (UK)', fontsize=13,
             fontweight='bold', pad=12, color=C['primary'])
ax.legend(handles=[mpatches.Patch(color=C['highlight'], label='Top 3'),
                   mpatches.Patch(color=C['secondary'], label='Lainnya')], loc='lower right')
savefig('plot2_top15_qty.png')

# ── Plot 3: Tren Bulanan Revenue & Transaksi ─────────────────
monthly = df.groupby('YearMonth').agg(
    Revenue=('Revenue','sum'), Transactions=('InvoiceNo','nunique')
).reset_index()
monthly['ym'] = monthly['YearMonth'].astype(str)
x = range(len(monthly))

fig, ax1 = plt.subplots(figsize=(13,5))
ax2 = ax1.twinx()
ax1.fill_between(x, monthly['Revenue']/1000, alpha=0.2, color=C['secondary'])
ax1.plot(x, monthly['Revenue']/1000, color=C['secondary'], lw=2.5, marker='o', ms=5, label='Revenue (£000)')
ax2.plot(x, monthly['Transactions'], color=C['highlight'], lw=2, ls='--', marker='s', ms=4, label='Transaksi')
ax1.set_xticks(list(x)); ax1.set_xticklabels(monthly['ym'], rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Revenue (£ ribuan)', color=C['secondary'], fontsize=11)
ax2.set_ylabel('Jumlah Transaksi', color=C['highlight'], fontsize=11)
ax1.set_title('Tren Revenue & Transaksi Bulanan UK', fontsize=13,
              fontweight='bold', pad=12, color=C['primary'])
l1, lb1 = ax1.get_legend_handles_labels(); l2, lb2 = ax2.get_legend_handles_labels()
ax1.legend(l1+l2, lb1+lb2, loc='upper left')
savefig('plot3_tren_bulanan.png')

# ── Plot 4: Distribusi Numerik 
fig, axes = plt.subplots(1, 2, figsize=(13,5))
axes[0].hist(df['Quantity'].clip(upper=100), bins=50, color=C['secondary'], edgecolor='white', lw=0.5)
axes[0].axvline(df['Quantity'].median(), color=C['highlight'], ls='--', lw=2,
                label=f"Median={df['Quantity'].median():.0f}")
axes[0].set_xlabel('Quantity (clip @100)'); axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Quantity', fontweight='bold', color=C['primary']); axes[0].legend()

axes[1].hist(df['UnitPrice'].clip(upper=20), bins=50, color=C['accent'], edgecolor='white', lw=0.5)
axes[1].axvline(df['UnitPrice'].median(), color=C['highlight'], ls='--', lw=2,
                label=f"Median=£{df['UnitPrice'].median():.2f}")
axes[1].set_xlabel('UnitPrice £ (clip @£20)'); axes[1].set_ylabel('Frekuensi')
axes[1].set_title('Distribusi Unit Price', fontweight='bold', color=C['primary']); axes[1].legend()
plt.suptitle('Distribusi Variabel Numerik — UK Data Bersih', fontsize=13,
             fontweight='bold', color=C['primary'], y=1.02)
plt.tight_layout()
savefig('plot4_distribusi_numerik.png')

# ── Plot 5: Pola Waktu Hari & Jam ────────────────────────────
day_order  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts  = df.groupby('DayOfWeek')['InvoiceNo'].nunique().reindex(day_order)
hour_counts = df.groupby('Hour')['InvoiceNo'].nunique()
peak_hour   = hour_counts.idxmax()

fig, axes = plt.subplots(1, 2, figsize=(14,5))
c_day = [C['highlight'] if v==day_counts.max() else C['secondary'] for v in day_counts]
axes[0].bar(day_counts.index, day_counts.values, color=c_day, edgecolor='white', width=0.7)
for i, v in enumerate(day_counts.values):
    axes[0].text(i, v+20, str(v), ha='center', fontsize=8, color=C['primary'])
axes[0].set_xticklabels(day_counts.index, rotation=30, ha='right')
axes[0].set_ylabel('Transaksi Unik'); axes[0].set_xlabel('Hari')
axes[0].set_title('Transaksi per Hari', fontweight='bold', color=C['primary'])

c_hour = [C['highlight'] if h==peak_hour else C['accent'] for h in hour_counts.index]
axes[1].bar(hour_counts.index, hour_counts.values, color=c_hour, edgecolor='white', width=0.8)
axes[1].set_xlabel('Jam (0–23)'); axes[1].set_ylabel('Transaksi Unik')
axes[1].set_title('Transaksi per Jam', fontweight='bold', color=C['primary'])
axes[1].axvline(peak_hour, color=C['highlight'], ls='--', lw=1.5, alpha=0.7, label=f'Peak: {peak_hour}:00')
axes[1].legend()
plt.suptitle('Pola Waktu Pembelian Pelanggan UK', fontsize=13,
             fontweight='bold', color=C['primary'], y=1.02)
plt.tight_layout()
savefig('plot5_pola_waktu.png')

# ── Plot 6: Pareto Revenue ───────────────────────────────────
top10r = df.groupby('Description')['Revenue'].sum().sort_values(ascending=False).head(10)
cumr   = top10r.cumsum() / top10r.sum() * 100
short10 = [l[:33]+'…' if len(l)>33 else l for l in top10r.index]

fig, ax1 = plt.subplots(figsize=(13,6))
ax2 = ax1.twinx()
ax1.bar(range(len(top10r)), top10r.values/1000, color=C['secondary'], edgecolor='white', width=0.6)
ax2.plot(range(len(top10r)), cumr.values, color=C['highlight'], lw=2.5, marker='o', ms=7, label='Cumulative %')
ax2.axhline(80, color=C['accent'], ls='--', lw=1.5, alpha=0.7, label='80% line')
ax1.set_xticks(range(len(top10r))); ax1.set_xticklabels(short10, rotation=35, ha='right', fontsize=8.5)
ax1.set_ylabel('Revenue (£ ribuan)', color=C['secondary'], fontsize=11)
ax2.set_ylabel('Cumulative Revenue (%)', color=C['highlight'], fontsize=11)
ax2.set_ylim(0,110)
ax1.set_title('Pareto Chart — Top 10 Produk berdasarkan Revenue', fontsize=13,
              fontweight='bold', pad=12, color=C['primary'])
ax2.legend(loc='center right')
savefig('plot6_pareto_revenue.png')

# ── Plot 7: Heatmap Hari × Jam ──────────────────────────────
pivot = df.pivot_table(values='InvoiceNo', index='DayOfWeek', columns='Hour',
                       aggfunc='nunique', fill_value=0).reindex(day_order)
fig, ax = plt.subplots(figsize=(15,5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, linecolor='white',
            annot=False, ax=ax, cbar_kws={'label':'Jumlah Transaksi'})
ax.set_title('Heatmap Aktivitas Transaksi — Hari × Jam (UK)',
             fontsize=13, fontweight='bold', pad=12, color=C['primary'])
ax.set_xlabel('Jam'); ax.set_ylabel('')
savefig('plot7_heatmap_hari_jam.png')

print(" Semua plot EDA selesai")

3. EDA — VISUALISASI
  Tersimpan: output_plots/plot1_preprocessing_funnel.png
  Tersimpan: output_plots/plot2_top15_qty.png
  Tersimpan: output_plots/plot3_tren_bulanan.png
  Tersimpan: output_plots/plot4_distribusi_numerik.png
  Tersimpan: output_plots/plot5_pola_waktu.png
  Tersimpan: output_plots/plot6_pareto_revenue.png
  Tersimpan: output_plots/plot7_heatmap_hari_jam.png
 Semua plot EDA selesai


## Transformasi Data (Basket Transaction)

Tahap ini data transaksi akan diubah ke bentuk basket transaction agar dapat diproses oleh algoritma association rule mining. 

Data kemudian ditransformasikan menjadi format biner, dimana setiap produk direpresentasikan dengan nilai 1 jika produk dibeli dan nilai 0 jika produk tidak dibeli. 

Transformasi ini diperlukan untuk mempermudah proses pencarian frequent itemset.

In [6]:
print("4. TRANSFORMASI DATA — BASKET TRANSACTION")

basket = df.groupby(['InvoiceNo','Description'])['Quantity'].sum().unstack(fill_value=0)
basket_bool = basket.astype(bool)
# hapus transaksi kosong 
basket_bool = basket_bool.loc[basket_bool.any(axis=1)]
# ubah True/False -> 1/0
basket_binary = basket_bool.astype(int)
# simpan CSV
basket_binary.to_csv('output_plots/basket_matrix_binary.csv')

print(f"Shape basket matrix : {basket_bool.shape}  ({basket_bool.shape[0]} transaksi × {basket_bool.shape[1]} produk)")
print(f"Density             : {basket_bool.values.mean()*100:.2f}%  (rata-rata item per transaksi: {basket_bool.sum(axis=1).mean():.1f})")
print(" Basket matrix binary tersimpan: output_plots/basket_matrix_binary.csv")

4. TRANSFORMASI DATA — BASKET TRANSACTION
Shape basket matrix : (18019, 3996)  (18019 transaksi × 3996 produk)
Density             : 0.66%  (rata-rata item per transaksi: 26.3)
 Basket matrix binary tersimpan: output_plots/basket_matrix_binary.csv


## Frequent Itemset Mining

Tahap ini dilakukan untuk menemukan kombinasi item yang sering muncul dalam transaksi pelanggan. 

Penelitian ini menggunakan tiga algoritma yaitu Apriori, ECLAT, dan FP-Growth. 

Setiap algoritma akan menggunakan nilai minimum support tertentu (min_sup = 0.01, 0.02, 0.05) untuk menghasilkan frequent itemset yang memenuhi batas frekuensi kemunculan.

In [7]:
print("5. FREQUENT ITEMSET MINING")

MIN_SUPPORTS = [0.01, 0.02, 0.05]
results = {}   # {algo: {min_sup: {'time':, 'mem':, 'n_itemset':, 'n_rules':, 'rules':}}}

for algo in ['Apriori', 'ECLAT', 'FP-Growth']:
    results[algo] = {}

# ─── Apriori 
print("\n[A] APRIORI")
for ms in MIN_SUPPORTS:
    tracemalloc.start()
    t0 = time.time()
    freq = apriori(basket_bool, min_support=ms, use_colnames=True, low_memory=True)
    rules = association_rules(freq, metric='lift', min_threshold=1.0) if len(freq) > 0 else pd.DataFrame()
    elapsed = time.time() - t0
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    results['Apriori'][ms] = {
        'time': elapsed, 'mem': peak/1024**2,
        'n_itemset': len(freq), 'n_rules': len(rules), 'rules': rules
    }
    print(f"  min_sup={ms:.2f} → {len(freq):>5} itemset | {len(rules):>5} rules | {elapsed:.2f}s | {peak/1024**2:.1f} MB")

# ─── ECLAT 
# Implementasi manual TID-intersection (tidak butuh pyECLAT)
print("\n[B] ECLAT  (implementasi TID-intersection manual)")
basket_indexed = basket_bool.reset_index(drop=True)
for ms in MIN_SUPPORTS:
    tracemalloc.start()
    t0 = time.time()
    frequent_eclat = run_eclat(basket_indexed, ms)
    freq_df = eclat_to_df(frequent_eclat)

    rules = association_rules(
        freq_df,
        metric='lift',
        min_threshold=1.0
    ) if len(freq_df) > 0 else pd.DataFrame()
    elapsed = time.time() - t0
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    n_itemset = len(frequent_eclat)
    results['ECLAT'][ms] = {
        'time': elapsed,
        'mem': peak/1024**2,
        'n_itemset': n_itemset,
        'n_rules': len(rules),
        'rules': rules
    }
    print(f"  min_sup={ms:.2f} → {n_itemset:>5} itemset | {len(rules):>5} rules | {elapsed:.2f}s | {peak/1024**2:.1f} MB")

# ─── FP-Growth
print("\n[C] FP-GROWTH")
for ms in MIN_SUPPORTS:
    tracemalloc.start()
    t0 = time.time()
    freq = fpgrowth(basket_bool, min_support=ms, use_colnames=True)
    rules = association_rules(freq, metric='lift', min_threshold=1.0) if len(freq) > 0 else pd.DataFrame()
    elapsed = time.time() - t0
    _, peak = tracemalloc.get_traced_memory(); tracemalloc.stop()
    results['FP-Growth'][ms] = {
        'time': elapsed, 'mem': peak/1024**2,
        'n_itemset': len(freq), 'n_rules': len(rules), 'rules': rules
    }
    print(f"  min_sup={ms:.2f} → {len(freq):>5} itemset | {len(rules):>5} rules | {elapsed:.2f}s | {peak/1024**2:.1f} MB")

5. FREQUENT ITEMSET MINING

[A] APRIORI
  min_sup=0.01 →  2158 itemset |  4382 rules | 3.66s | 8.6 MB
  min_sup=0.02 →   401 itemset |   218 rules | 1.16s | 8.5 MB
  min_sup=0.05 →    34 itemset |     0 rules | 0.35s | 8.5 MB

[B] ECLAT  (implementasi TID-intersection manual)
  min_sup=0.01 →  2158 itemset |  4382 rules | 15.37s | 41.3 MB
  min_sup=0.02 →   401 itemset |   218 rules | 6.02s | 18.9 MB
  min_sup=0.05 →    34 itemset |     0 rules | 6.36s | 3.1 MB

[C] FP-GROWTH
  min_sup=0.01 →  2158 itemset |  4382 rules | 77.24s | 99.3 MB
  min_sup=0.02 →   401 itemset |   218 rules | 25.20s | 48.4 MB
  min_sup=0.05 →    34 itemset |     0 rules | 5.16s | 4.9 MB


## Analisis Komparatif

In [8]:
# 6. TABEL KOMPARATIF PERFORMA
print("6. TABEL KOMPARATIF PERFORMA")

rows = []
for ms in MIN_SUPPORTS:
    for algo in ['Apriori', 'ECLAT', 'FP-Growth']:
        r = results[algo][ms]
        rows.append({
            'Algoritma': algo,
            'min_support': ms,
            'Waktu (s)': round(r['time'], 3),
            'Memori (MB)': round(r['mem'], 2),
            'Frequent Itemset': r['n_itemset'],
            'Association Rules': r['n_rules'],
        })

perf_df = pd.DataFrame(rows)
print(perf_df.to_string(index=False))
perf_df.to_csv('output_plots/tabel_komparatif_performa.csv', index=False)
print("\n Tersimpan: output_plots/tabel_komparatif_performa.csv")

# 7. TOP-10 ASSOCIATION RULES 
print("7. TOP-10 ASSOCIATION RULES (Apriori, min_sup=0.02)") 
best_rules = results['Apriori'][0.02]['rules']
if len(best_rules) > 0:
    top10_rules = best_rules.sort_values('lift', ascending=False).head(10)
    top10_rules = top10_rules[
        ['antecedents','consequents','support','confidence','lift']
    ].copy()
    top10_rules['antecedents'] = top10_rules['antecedents'].apply(
        lambda x: ', '.join(list(x))
    )
    top10_rules['consequents'] = top10_rules['consequents'].apply(
        lambda x: ', '.join(list(x))
    )
    top10_rules = top10_rules.round({
        'support':4,
        'confidence':4,
        'lift':4
    })
    print(top10_rules.to_string(index=False))
    top10_rules.to_csv(
        'output_plots/top10_association_rules_apriori_0,02.csv',
        index=False
    )
    print("\nTersimpan: output_plots/top10_association_rules_apriori_0,02.csv")
else:
    print(" Tidak ada rules yang dihasilkan pada setting ini.")
    top10_rules = pd.DataFrame()

6. TABEL KOMPARATIF PERFORMA
Algoritma  min_support  Waktu (s)  Memori (MB)  Frequent Itemset  Association Rules
  Apriori         0.01      3.656         8.58              2158               4382
    ECLAT         0.01     15.371        41.27              2158               4382
FP-Growth         0.01     77.236        99.29              2158               4382
  Apriori         0.02      1.165         8.51               401                218
    ECLAT         0.02      6.018        18.91               401                218
FP-Growth         0.02     25.196        48.37               401                218
  Apriori         0.05      0.345         8.48                34                  0
    ECLAT         0.05      6.361         3.06                34                  0
FP-Growth         0.05      5.155         4.91                34                  0

 Tersimpan: output_plots/tabel_komparatif_performa.csv
7. TOP-10 ASSOCIATION RULES (Apriori, min_sup=0.02)
                       

In [9]:
# 8. VISUALISASI KOMPARASI ALGORITMA
print("8. VISUALISASI KOMPARASI ALGORITMA")

algo_colors = {'Apriori': C['apriori'], 'ECLAT': C['eclat'], 'FP-Growth': C['fpgrowth']}
sup_labels  = [str(s) for s in MIN_SUPPORTS]

# ── Plot 8a: Waktu Eksekusi ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9,5))
for algo in ['Apriori', 'ECLAT', 'FP-Growth']:
    times = [results[algo][ms]['time'] for ms in MIN_SUPPORTS]
    ax.plot(sup_labels, times, marker='o', ms=8, lw=2.5,
            color=algo_colors[algo], label=algo)
    for i, (s, t) in enumerate(zip(sup_labels, times)):
        ax.annotate(f'{t:.2f}s', (s, t), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8, color=algo_colors[algo])
ax.set_xlabel('Minimum Support', fontsize=11)
ax.set_ylabel('Waktu Eksekusi (detik)', fontsize=11)
ax.set_title('Komparasi Waktu Eksekusi Algoritma', fontsize=13,
             fontweight='bold', pad=12, color=C['primary'])
ax.legend(); ax.set_yscale('log')
savefig('plot8a_waktu_eksekusi.png')

# ── Plot 8b: Memori ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(MIN_SUPPORTS)); w = 0.25
for i, algo in enumerate(['Apriori', 'ECLAT', 'FP-Growth']):
    mems = [results[algo][ms]['mem'] for ms in MIN_SUPPORTS]
    bars = ax.bar(x + i*w, mems, width=w, color=algo_colors[algo],
                  edgecolor='white', label=algo)
    for b, v in zip(bars, mems):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.3,
                f'{v:.1f}', ha='center', fontsize=7.5)
ax.set_xticks(x + w); ax.set_xticklabels([f'min_sup={s}' for s in MIN_SUPPORTS])
ax.set_ylabel('Peak Memory (MB)', fontsize=11)
ax.set_title('Komparasi Konsumsi Memori Algoritma', fontsize=13,
             fontweight='bold', pad=12, color=C['primary'])
ax.legend()
savefig('plot8b_memori.png')

# ── Plot 8c: Jumlah Frequent Itemset ────────────────────────
fig, ax = plt.subplots(figsize=(9,5))
for algo in ['Apriori', 'ECLAT', 'FP-Growth']:
    nitems = [results[algo][ms]['n_itemset'] for ms in MIN_SUPPORTS]
    ax.plot(sup_labels, nitems, marker='s', ms=8, lw=2.5,
            color=algo_colors[algo], label=algo)
ax.set_xlabel('Minimum Support', fontsize=11)
ax.set_ylabel('Jumlah Frequent Itemset', fontsize=11)
ax.set_title('Jumlah Frequent Itemset per Algoritma', fontsize=13,
             fontweight='bold', pad=12, color=C['primary'])
ax.legend()
savefig('plot8c_jumlah_itemset.png')

# ── Plot 8d: Jumlah Association Rules ───────────────────────
fig, ax = plt.subplots(figsize=(9,5))
for algo in ['Apriori', 'FP-Growth']:     # ECLAT tidak generate rules via mlxtend
    nrules = [results[algo][ms]['n_rules'] for ms in MIN_SUPPORTS]
    ax.plot(sup_labels, nrules, marker='^', ms=8, lw=2.5,
            color=algo_colors[algo], label=algo)
ax.set_xlabel('Minimum Support', fontsize=11)
ax.set_ylabel('Jumlah Association Rules', fontsize=11)
ax.set_title('Jumlah Association Rules per Algoritma', fontsize=13,
             fontweight='bold', pad=12, color=C['primary'])
ax.legend()
savefig('plot8d_jumlah_rules.png')

# ── Plot 9: Top-10 Rules — Lift Bar Chart ───────────────────
if len(top10_rules) > 0:
    fig, ax = plt.subplots(figsize=(12,6))
    labels_r = [f"{row['antecedents'][:25]}… → {row['consequents'][:20]}…"
                if len(row['antecedents']) > 25 else f"{row['antecedents']} → {row['consequents']}"
                for _, row in top10_rules.iterrows()]
    bars = ax.barh(labels_r[::-1], top10_rules['lift'].values[::-1],
                   color=C['green'], edgecolor='white', height=0.65)
    for b, v in zip(bars, top10_rules['lift'].values[::-1]):
        ax.text(b.get_width()+0.05, b.get_y()+b.get_height()/2,
                f'{v:.2f}', va='center', fontsize=8.5, color=C['primary'])
    ax.axvline(1, color=C['highlight'], ls='--', lw=1.5, alpha=0.7, label='Lift = 1')
    ax.set_xlabel('Lift Value', fontsize=11)
    ax.set_title('Top-10 Association Rules berdasarkan Lift Tertinggi\n(FP-Growth, min_sup=0.02)',
                 fontsize=12, fontweight='bold', pad=12, color=C['primary'])
    ax.legend()
    savefig('plot9_top10_rules_lift.png')

8. VISUALISASI KOMPARASI ALGORITMA
  Tersimpan: output_plots/plot8a_waktu_eksekusi.png
  Tersimpan: output_plots/plot8b_memori.png
  Tersimpan: output_plots/plot8c_jumlah_itemset.png
  Tersimpan: output_plots/plot8d_jumlah_rules.png
  Tersimpan: output_plots/plot9_top10_rules_lift.png
